# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MA-1305/Project_Mahin-1305/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

print(model)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)


## Method Choice

I selected Random Forest because the task contains several numeric and categorical page-level signals that may interact in non-linear ways.

The model can estimate the probability that a page belongs to the declining class, which can then be used to rank pages for content review. The model is used as decision support rather than proof that a refresh will cause improved performance.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.datasets import load_iris


In [20]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# Make sure the repository exists
repo_path = Path("/content/Project_Mahin-1305")

if not repo_path.exists():
    !git clone https://github.com/MA-1305/Project_Mahin-1305.git /content/Project_Mahin-1305

# Load starter dataset
data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

# Create proxy target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Features from W03
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining_label"]

# Client-holdout split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=df["client_id"]
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print(
    "Training clients:",
    df.iloc[train_idx]["client_id"].nunique()
)

print(
    "Testing clients:",
    df.iloc[test_idx]["client_id"].nunique()
)

Training data: (23837, 27)
Testing data: (6163, 27)
Training clients: 25
Testing clients: 7


## Split Design

I use a client-holdout split so that a client is entirely in either the training or testing set.

This is more honest for the task because the model is evaluated on clients that were not used during training. The same split is used for the baseline comparison and the Random Forest model.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score,
    average_precision_score,
    roc_auc_score
)

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Complete model pipeline
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# Train
model_pipeline.fit(X_train, y_train)

# Model probabilities
model_prob = model_pipeline.predict_proba(X_test)[:, 1]

# Predictions
model_pred = (model_prob >= 0.5).astype(int)

# Precision@50
k = 50

top_50_model_idx = model_prob.argsort()[::-1][:k]

model_precision_at_50 = y_test.iloc[top_50_model_idx].mean()

# Other metrics
model_auc = roc_auc_score(y_test, model_prob)
model_average_precision = average_precision_score(
    y_test,
    model_prob
)

print("Random Forest ROC AUC:",
      round(model_auc, 3))

print("Random Forest Average Precision:",
      round(model_average_precision, 3))

print("Random Forest Precision@50:",
      round(model_precision_at_50, 3))

Random Forest ROC AUC: 0.613
Random Forest Average Precision: 0.601
Random Forest Precision@50: 0.62


In [22]:
# Recreate the Week-4 transparent baseline on the test rows
import numpy as np

baseline_test = df.iloc[test_idx].copy()

baseline_test["visibility_score"] = (
    np.log1p(baseline_test["impressions_90d"])
    .rank(pct=True)
)

baseline_test["freshness_risk_score"] = (
    baseline_test["days_since_last_update"]
    .clip(lower=0)
    .rank(pct=True)
)

baseline_test["position_opportunity_score"] = (
    1 - (
        (baseline_test["avg_position"]
         .clip(lower=1, upper=20) - 1) / 19
    )
).fillna(0)

baseline_test["depth_gap_score"] = (
    (1200 - baseline_test["word_count"].fillna(0))
    / 1200
).clip(lower=0, upper=1)

baseline_test["baseline_refresh_score"] = 100 * (
    0.40 * baseline_test["visibility_score"]
    + 0.30 * baseline_test["freshness_risk_score"]
    + 0.25 * baseline_test["position_opportunity_score"]
    + 0.05 * baseline_test["depth_gap_score"]
)

baseline_order = (
    baseline_test["baseline_refresh_score"]
    .sort_values(ascending=False)
    .index
)

top_50_baseline_idx = baseline_order[:50]

baseline_precision_at_50 = (
    baseline_test.loc[
        top_50_baseline_idx,
        "is_declining_label"
    ].mean()
)

comparison = pd.DataFrame({
    "Model": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

print(comparison)

             Model  Precision@50
0  Week-4 Baseline          0.32
1    Random Forest          0.62


## Model Training and Baseline Comparison

The Random Forest is trained using the same client-holdout split as the baseline comparison.

Precision@50 is used because the practical goal is to rank pages for review. The comparison measures how many positive pages appear in the first 50 recommendations.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create model action queue

from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report")
print(
    classification_report(
        y_test,
        model_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, model_pred))

# Feature importance
feature_names = (
    model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importance = (
    model_pipeline
    .named_steps["model"]
    .feature_importances_
)

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\nTop 15 Feature Importances")
print(importance.head(15))

Classification Report
              precision    recall  f1-score   support

           0       0.58      0.48      0.52      3014
           1       0.57      0.68      0.62      3149

    accuracy                           0.58      6163
   macro avg       0.58      0.58      0.57      6163
weighted avg       0.58      0.58      0.57      6163


Confusion Matrix
[[1433 1581]
 [1018 2131]]

Top 15 Feature Importances
                       Feature  Importance
5         num__impressions_90d    0.102565
14           num__avg_position    0.095060
9   num__days_with_impressions    0.080565
11       num__content_age_days    0.071222
4              num__char_count    0.055284
3              num__word_count    0.054175
7            num__sessions_90d    0.053693
16            num__scroll_rate    0.050512
10     num__days_with_sessions    0.050437
13                    num__ctr    0.050180
6              num__clicks_90d    0.039417
0           num__search_volume    0.031921
1             num__

## Error Analysis

The classification report and confusion matrix show where the model makes incorrect predictions on the held-out clients.

The feature importance table shows which transformed features contribute most to the Random Forest predictions. These results are descriptive and should be treated as decision support rather than proof that any feature causes decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.